# Component 3 — Environmental Lag Analysis

This notebook tests whether chlorophyll-a and sea-surface temperature observed in earlier months are associated with catch outcomes in a later month for the Western Province study data. It consumes the complete integrated artifact produced by `03_data_integration.ipynb`.

## Analytical safeguards

- **Lag direction:** lag `k` uses environment at month `t-k` with catch at month `t`; future information is never shifted backward.
- **Temporal unit:** records are aggregated to calendar months before lagging so dense sampling does not create thousands of false independent time points.
- **Primary outcome:** median catch per record is used as a robust catch-availability proxy because reliable effort is unavailable. Total catch is secondary and remains effort-sensitive.
- **Seasonality:** both raw correlations and calendar-month-median anomalies are reported because shared monsoon seasonality can produce misleading raw lag relationships.
- **Missing months:** a complete monthly index is created, but missing observations are not interpolated or backfilled.
- **Inference boundary:** correlations and p-values are exploratory, not causal. Final lag selection for machine learning must occur inside training data only.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4','scipy':'scipy>=1.10,<2'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('Lag-analysis dependencies are available.')

In [ ]:
from __future__ import annotations
import json, os, sys
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
try:
    from IPython.display import display
except ImportError:
    display=print
pd.set_option('display.max_columns',80); plt.style.use('seaborn-v0_8-whitegrid')
COLORS={'catch':'#4e79a7','chl':'#59a14f','sst':'#f28e2b','accent':'#8f63b8'}
MAX_LAG=6

## 1. Locate and load the integrated complete-case artifact

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
PROCESSED_DIR=Path(os.getenv('FISHGUARD_C3_PROCESSED_DIR',str(ROOT/'data'/'processed'))).resolve()
EVIDENCE_DIR=Path(os.getenv('FISHGUARD_C3_LAG_OUTPUT',str(ROOT/'outputs'/'audit'/'lag_analysis'))).resolve()
FIGURE_DIR=Path(os.getenv('FISHGUARD_C3_LAG_FIGURES',str(ROOT/'outputs'/'figures'/'lag_analysis'))).resolve()
for directory in [EVIDENCE_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
DATA_PATH=PROCESSED_DIR/'fisheries_environment_integrated_complete.parquet'
if not DATA_PATH.is_file(): raise FileNotFoundError(f'Missing {DATA_PATH}. Run 03_data_integration.ipynb first.')
df=pd.read_parquet(DATA_PATH)
required={'event_date','catch_weight_kg','chlorophyll_a_mg_m3','sea_surface_temperature_c','environment_complete','source_row_number'}
missing=required-set(df.columns)
if missing: raise KeyError(f'Missing integrated fields: {sorted(missing)}')
assert df.environment_complete.fillna(False).all(), 'The complete-case artifact contains incomplete environmental matches.'
df['event_date']=pd.to_datetime(df.event_date,errors='coerce')
numeric=['catch_weight_kg','chlorophyll_a_mg_m3','sea_surface_temperature_c']
for column in numeric: df[column]=pd.to_numeric(df[column],errors='coerce')
analysis=df.dropna(subset=['event_date',*numeric]).copy()
analysis=analysis.loc[(analysis.catch_weight_kg>=0)&(analysis.chlorophyll_a_mg_m3>=0)]
analysis['month_start']=analysis.event_date.dt.to_period('M').dt.to_timestamp()
print('Runtime:','Google Colab' if IN_COLAB else 'Local Windows')
print('Input:',DATA_PATH); print('Eligible rows:',f'{len(analysis):,}')
print('Coverage:',analysis.event_date.min().date(),'to',analysis.event_date.max().date())

## 2. Construct a continuous monthly series

Monthly median catch is the primary proxy. Monthly total catch and record count are retained to reveal sampling-effort sensitivity. Environmental values are summarized by their monthly medians.

In [ ]:
monthly=(analysis.groupby('month_start').agg(
    records=('source_row_number','size'),
    total_catch_weight_kg=('catch_weight_kg','sum'),
    median_catch_weight_kg=('catch_weight_kg','median'),
    mean_catch_weight_kg=('catch_weight_kg','mean'),
    median_chlorophyll_a_mg_m3=('chlorophyll_a_mg_m3','median'),
    median_sst_c=('sea_surface_temperature_c','median')).sort_index())
full_index=pd.date_range(monthly.index.min(),monthly.index.max(),freq='MS',name='month_start')
monthly=monthly.reindex(full_index)
monthly['observed_month']=monthly.records.notna()
monthly['calendar_month']=monthly.index.month
monthly['log1p_median_catch']=np.log1p(monthly.median_catch_weight_kg.clip(lower=0))
monthly['log1p_total_catch']=np.log1p(monthly.total_catch_weight_kg.clip(lower=0))
monthly.reset_index().to_csv(EVIDENCE_DIR/'monthly_lag_base.csv',index=False)
coverage=pd.DataFrame([
 {'metric':'eligible_integrated_rows','value':len(analysis)},
 {'metric':'calendar_months_in_range','value':len(monthly)},
 {'metric':'observed_months','value':int(monthly.observed_month.sum())},
 {'metric':'missing_months','value':int((~monthly.observed_month).sum())},
 {'metric':'maximum_lag_months','value':MAX_LAG}])
coverage.to_csv(EVIDENCE_DIR/'lag_analysis_overview.csv',index=False)
display(coverage); display(monthly.head())

In [ ]:
fig,axes=plt.subplots(4,1,figsize=(14,12),sharex=True)
axes[0].plot(monthly.index,monthly.median_catch_weight_kg,color=COLORS['catch']); axes[0].set_ylabel('Median catch (kg)')
axes[1].plot(monthly.index,monthly.total_catch_weight_kg,color=COLORS['accent']); axes[1].set_ylabel('Total catch (kg)')
axes[2].plot(monthly.index,monthly.median_chlorophyll_a_mg_m3,color=COLORS['chl']); axes[2].set_ylabel('Median chlorophyll-a')
axes[3].plot(monthly.index,monthly.median_sst_c,color=COLORS['sst']); axes[3].set_ylabel('Median SST (°C)'); axes[3].set_xlabel('Catch month')
fig.suptitle('Monthly catch and environmental signals — Western Province study data')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'monthly_lag_series.png',dpi=200,bbox_inches='tight'); plt.show()

## 3. Remove recurring calendar-month seasonality

An anomaly is the monthly value minus the median for the same calendar month across years. This does not prove causality, but it helps distinguish lag relationships from simple shared seasonality.

In [ ]:
signal_columns=['log1p_median_catch','log1p_total_catch','median_chlorophyll_a_mg_m3','median_sst_c']
for column in signal_columns:
    seasonal_median=monthly.groupby('calendar_month')[column].transform('median')
    monthly[f'{column}_anomaly']=monthly[column]-seasonal_median
anomaly_columns=[f'{column}_anomaly' for column in signal_columns]
monthly.reset_index().to_csv(EVIDENCE_DIR/'monthly_lag_base.csv',index=False)
assert monthly.loc[~monthly.observed_month,signal_columns+anomaly_columns].isna().all().all(), 'Missing months were unexpectedly filled.'
display(monthly[signal_columns+anomaly_columns].describe().T)

## 4. Test environmental leads from 0 to 6 months

In [ ]:
targets={'median_catch':'log1p_median_catch','total_catch':'log1p_total_catch'}
signals={'chlorophyll_a':'median_chlorophyll_a_mg_m3','sst':'median_sst_c'}
results=[]
for adjustment in ['raw','seasonally_adjusted']:
    for target_name,target_column in targets.items():
        y_column=target_column if adjustment=='raw' else f'{target_column}_anomaly'
        for signal_name,signal_column in signals.items():
            x_base=signal_column if adjustment=='raw' else f'{signal_column}_anomaly'
            for lag in range(MAX_LAG+1):
                pair=pd.concat([monthly[y_column],monthly[x_base].shift(lag)],axis=1).dropna()
                pair.columns=['catch','environment']
                rho,p_value=spearmanr(pair.catch,pair.environment) if len(pair)>=3 else (np.nan,np.nan)
                results.append({'adjustment':adjustment,'target':target_name,'environment':signal_name,'lag_months':lag,'n_pairs':len(pair),'spearman_rho':rho,'p_value_unadjusted':p_value})
lag_results=pd.DataFrame(results)
def benjamini_hochberg(values):
    values=np.asarray(values,dtype=float); adjusted=np.full(len(values),np.nan); valid=np.flatnonzero(np.isfinite(values))
    if not len(valid): return adjusted
    order=valid[np.argsort(values[valid])]; ranked=values[order]*len(order)/np.arange(1,len(order)+1)
    adjusted[order]=np.minimum.accumulate(ranked[::-1])[::-1].clip(0,1); return adjusted
lag_results['q_value_bh']=benjamini_hochberg(lag_results.p_value_unadjusted)
lag_results['absolute_rho']=lag_results.spearman_rho.abs()
lag_results.to_csv(EVIDENCE_DIR/'lag_correlations.csv',index=False)
ranked=lag_results.sort_values(['adjustment','target','absolute_rho'],ascending=[True,True,False])
ranked.to_csv(EVIDENCE_DIR/'lag_rankings.csv',index=False)
display(ranked.groupby(['adjustment','target'],group_keys=False).head(5))

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10),sharex=True,sharey=True)
for row,adjustment in enumerate(['raw','seasonally_adjusted']):
    for col,target in enumerate(['median_catch','total_catch']):
        ax=axes[row,col]
        for signal,color in [('chlorophyll_a',COLORS['chl']),('sst',COLORS['sst'])]:
            subset=lag_results.query('adjustment==@adjustment and target==@target and environment==@signal')
            ax.plot(subset.lag_months,subset.spearman_rho,marker='o',label=signal,color=color)
        ax.axhline(0,color='black',linewidth=.8); ax.set_xticks(range(MAX_LAG+1)); ax.set_ylim(-1,1)
        ax.set_title(f"{adjustment.replace('_',' ').title()} — {target.replace('_',' ')}")
        ax.set_xlabel('Environmental lead (months)'); ax.set_ylabel('Spearman rho'); ax.legend()
fig.suptitle('Environmental lag-correlation profiles')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'lag_correlation_profiles.png',dpi=200,bbox_inches='tight'); plt.show()

## 5. Inspect the strongest seasonally adjusted candidates

Only positive lead months are candidates for forecasting. These full-period rankings are exploratory; they must not be copied blindly into the final model.

In [ ]:
candidates=(lag_results.query("adjustment=='seasonally_adjusted' and target=='median_catch' and lag_months>0")
            .sort_values('absolute_rho',ascending=False).head(4).reset_index(drop=True))
candidates.insert(0,'candidate_rank',np.arange(1,len(candidates)+1))
candidates.to_csv(EVIDENCE_DIR/'candidate_environmental_lags.csv',index=False)
fig,axes=plt.subplots(2,2,figsize=(13,10)); axes=np.asarray(axes).ravel()
for ax,(_,candidate) in zip(axes,candidates.iterrows()):
    signal_column=signals[candidate.environment]+'_anomaly'
    pair=pd.concat([monthly['log1p_median_catch_anomaly'],monthly[signal_column].shift(int(candidate.lag_months))],axis=1).dropna()
    pair.columns=['catch_anomaly','environment_anomaly']
    ax.scatter(pair.environment_anomaly,pair.catch_anomaly,alpha=.7,color=COLORS['chl'] if candidate.environment=='chlorophyll_a' else COLORS['sst'])
    if len(pair)>1:
        slope,intercept=np.polyfit(pair.environment_anomaly,pair.catch_anomaly,1); x=np.linspace(pair.environment_anomaly.min(),pair.environment_anomaly.max(),100); ax.plot(x,slope*x+intercept,color='black',linewidth=1)
    ax.set_title(f"{candidate.environment}, lead {int(candidate.lag_months)} month(s)\nρ={candidate.spearman_rho:.2f}, n={int(candidate.n_pairs)}")
    ax.set_xlabel('Environmental anomaly'); ax.set_ylabel('Log median-catch anomaly')
for ax in axes[len(candidates):]: ax.set_visible(False)
fig.suptitle('Strongest exploratory lag candidates after seasonal adjustment')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'candidate_lag_scatterplots.png',dpi=200,bbox_inches='tight'); plt.show()
display(candidates)

## 6. Stability by time period

In [ ]:
observed_years=sorted(monthly.index[monthly.observed_month].year.unique())
split_year=observed_years[len(observed_years)//2]
stability=[]
for _,candidate in candidates.iterrows():
    signal_column=signals[candidate.environment]+'_anomaly'; lag=int(candidate.lag_months)
    pair=pd.concat([monthly['log1p_median_catch_anomaly'],monthly[signal_column].shift(lag)],axis=1).dropna(); pair.columns=['catch','environment']
    for period,mask in [('early',pair.index.year<split_year),('late',pair.index.year>=split_year)]:
        part=pair.loc[mask]; rho,p=spearmanr(part.catch,part.environment) if len(part)>=3 else (np.nan,np.nan)
        stability.append({'candidate_rank':int(candidate.candidate_rank),'environment':candidate.environment,'lag_months':lag,'period':period,'year_rule':f'< {split_year}' if period=='early' else f'>= {split_year}','n_pairs':len(part),'spearman_rho':rho,'p_value_unadjusted':p})
stability=pd.DataFrame(stability)
stability.to_csv(EVIDENCE_DIR/'candidate_lag_stability.csv',index=False)
display(stability)

## 7. Save viva findings and reproducibility manifest

In [ ]:
best=candidates.iloc[0] if len(candidates) else None
findings=pd.DataFrame([
 {'finding':'Temporal unit','result':'Calendar month'},
 {'finding':'Primary catch outcome','result':'Median catch weight per record (log1p for correlation)'},
 {'finding':'Lag direction','result':'Environment at t-k compared with catch at t'},
 {'finding':'Maximum environmental lead','result':f'{MAX_LAG} months'},
 {'finding':'Observed monthly coverage','result':f"{int(monthly.observed_month.sum())} of {len(monthly)} calendar months"},
 {'finding':'Strongest seasonally adjusted positive-lag candidate','result':('None' if best is None else f"{best.environment}, {int(best.lag_months)} month(s), Spearman rho={best.spearman_rho:.3f}")},
 {'finding':'Effort limitation','result':'Total catch is secondary because vessel/trip/fishing-effort exposure is unavailable'},
 {'finding':'Modelling safeguard','result':'Re-estimate and select lag features inside each training fold; do not use full-period rankings as final feature selection'},
 {'finding':'Interpretation limit','result':'Exploratory association only; seasonality, sampling, location, species, gear and autocorrelation may confound results'}])
findings.to_csv(EVIDENCE_DIR/'lag_analysis_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','input':DATA_PATH.name,'eligible_rows':len(analysis),'temporal_unit':'calendar month','max_lag_months':MAX_LAG,'lag_definition':'environment at t-k paired with catch at t','primary_outcome':'monthly median catch weight per record','seasonal_adjustment':'subtract calendar-month median across years','missing_month_policy':'retain as missing; no interpolation or backfill','source_artifact_modified':False,'scope':'Western Province study data; exploratory associations only','figures':sorted(p.name for p in FIGURE_DIR.glob('*.png')),'evidence':sorted(p.name for p in EVIDENCE_DIR.glob('*.csv'))}
with (EVIDENCE_DIR/'lag_analysis_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(findings); print('Evidence:',EVIDENCE_DIR); print('Figures:',FIGURE_DIR)

## Viva interpretation

A positive lag means the environmental measurement occurs **before** the catch outcome. Raw lag peaks can reflect the annual monsoon cycle, so the seasonally adjusted result is more informative for candidate generation. A strong full-period coefficient is still not proof of causation or guaranteed forecasting value. The subsequent feature-engineering and modelling stages must preserve chronological order and select lags using training data only.